In [1]:
import numpy as np
import pandas as pd
import os
import joblib
import pickle
import math
import ast
from scipy.stats import median_abs_deviation, hypergeom, mannwhitneyu
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
# Saving plots with editable text
plt.rcParams['pdf.fonttype'] = 42  # TrueType fonts (editable text)

In [2]:
import sys
# Ensure this analysis directory is importable regardless of kernel CWD
_here = '/projects/bhdw/asachan/methods/FIREFate/multiome_dynamic_regulation/py_scripts/analysis'
if _here not in sys.path:
    sys.path.insert(0, _here)

import dictys
from utils_custom import *
from pseudotime_curves import *
from episodic_dynamics import *
from config import *

In [ ]:
import importlib
import firefate.utils.plots, firefate.utils.custom
import temporal_clustering
# reload firefate helpers first, then temporal_clustering so it re-binds fresh names
importlib.reload(firefate.utils.plots)
importlib.reload(firefate.utils.custom)
importlib.reload(temporal_clustering)
from temporal_clustering import FateFrequency, TFForceWaves, WaveValidation

In [4]:
config = Config()

In [5]:
# Load data
dictys_dynamic_object = dictys.net.dynamic_network.from_file('/work/nvme/bhdw/asachan/data_files/firefate/bcell/outs/dynamic.h5')

## Cell state distributions over pseudotime (fate frequency trajectory)

In [6]:
PB_fate_window_indices = [1] + list(range(97, 3, -1)) + [0] + list(range(98, 147, 1)) + [2]
GC_fate_window_indices = [1] + list(range(97, 3, -1)) + [0] + list(range(147, 193, 1)) + [3]
PB_post_bifurcation_window_indices = [0] + list(range(98, 147, 1)) + [2]
GC_post_bifurcation_window_indices = [0] + list(range(147, 193, 1)) + [3]

In [7]:
# Define distinct colors for better visibility
colors_cell_count = {
    'ActB-1': '#87CEFA',     # lightskyblue
    'ActB-2': '#1E90FF',     # dodgerblue
    'ActB-3': '#00008B',     # darkblue
    'ActB-4': '#9370DB',     # mediumorchid
    'GC-1': '#7BDE7B',       # custom light green
    'GC-2': '#008000',       # green
    'PB-2': '#BB3636',       # custom red
    'earlyActB': '#008080',   # teal
    'earlyPB': '#F08080'   # lightcoral
}

In [ ]:
# Class 1: cell-state fate frequencies + extrema / termination pseudotimes
ff = FateFrequency(
    dictys_dynamic_object,
    cell_labels=config.CELL_LABELS,
    colors=colors_cell_count,
    trajectory_range=(0, 2),
)
pseudotime_values_of_windows = ff.pseudotime_values_of_windows

#### Stacked bar plot of cell states across pseudotime bins

In [ ]:
# sanity check: states x windows count table (ActB-1 / earlyActB dropped internally)
display(ff.state_count_per_window.head())

In [ ]:
# Stacked bar plot of average cell-state composition over binned windows (PB branch)
fig, ax = ff.plot_stacked_bars(PB_post_bifurcation_window_indices, n_bins=8, figsize=(6, 4))
# fig.savefig(os.path.join(config.OUTPUT_FOLDER, 'fig4_state_count_PB_bins.pdf'), dpi=1200)
plt.show()

### State composition curves

In [ ]:
# State composition curves (PB branch)
fig, ax = ff.plot_composition_curves(PB_post_bifurcation_window_indices, figsize=(15, 8), xlabel='PB branch')
# fig.savefig('.../day_count_per_window_PB.pdf', bbox_inches='tight', dpi=100, format='pdf')
plt.show()

In [ ]:
# Per-state extrema (maxima/minima) of the count trajectory
extrema_info, extrema_pseudotimes = ff.find_extrema(
    PB_post_bifurcation_window_indices, prominence=10, distance=3
)

print("=" * 70)
print("EXTREMA PSEUDOTIMES DICTIONARY")
print("=" * 70)
for state, vals in extrema_pseudotimes.items():
    print(f"\n'{state}': {{")
    print(f"    'maxima': {vals['maxima']},")
    print(f"    'minima': {vals['minima']},")
    print(f"    'all': {vals['all']},")
    print("}")

fig, ax = ff.plot_extrema(PB_post_bifurcation_window_indices, prominence=10, distance=3)
plt.show()

# TF Forces

In [ ]:
# Class 2: TF forces over pseudotime (expression / regulation curves cached internally)
waves = TFForceWaves(
    dictys_dynamic_object,
    trajectory_range=(0, 2),
    num_points=100,
    dist=0.0005,
    sparsity=0.01,
)

In [ ]:
# Plot expression trajectories
# Highlight specific genes
#genes_of_interest_pb = ['CREB3L2','BACH2']
#genes_of_interest_pb = ['XBP1','PRDM1']
genes_of_interest_gc = ['PAX5', 'NFKB1', 'CREB3L2']
#colors_pb = ['blueviolet','slateblue']
colors_gc = ['limegreen', 'green', 'olivedrab']

fig, ax = waves.plot_expression(genes_of_interest_gc, colors_gc, ylabel='Log (CPM)')
# fig.savefig(os.path.join(config.OUTPUT_FOLDER, 'gc_2nd_wave_tfs_expression.pdf'), dpi=300)
plt.show()

In [ ]:
# Plot regulation trajectories
fig, ax = waves.plot_regulation(genes_of_interest_gc, colors_gc, ylabel='Log (target counts)')
# fig.savefig(os.path.join(config.OUTPUT_FOLDER, 'gc_2nd_wave_tfs_regulation.pdf'), dpi=300)
plt.show()

#### Prioritized links

In [19]:
PB_links_plotting = [('BACH2','XBP1'),('IRF4','CDK6'),('CREB3L2','FNDC3A'),('RUNX2','PRDM1'),('CREB3L2','MZB1'),('CREB3L2','RUNX2'),('CREB3L2','TXNDC5'),('TCF12','SEL1L3'),('IRF4','PRDM1'),
                    ('IRF4','ELL2'),('XBP1','HSP90B1'),('XBP1','PPIB'),('XBP1','TXNDC11'),('PRDM1','IRF4'),('CREB3L2','FNDC3B'),('PRDM1','RUNX2')]

In [20]:
GC_links_plotting = [('ARID5B','PIKFYVE'),('ARID5B','PDE4D'),('IRF4','PAX5'),('BATF','PPIB'),('NFKB1','AFF3'),('IRF4','AFF3'),('BACH2','MZB1'),('PAX5','GLCCI1'),
                     ('PAX5','PRDM1'),('CREB3L2','PAX5'),('NFKB1','PAX5')]

In [ ]:
# Compute beta / TF-expression / force curves for the prioritized links
force_curves = waves.compute_forces(PB_links_plotting, varname='w_in')
dtime = waves.dtime

In [ ]:
display(waves.beta_curves.head())

In [ ]:
display(waves.force_curves.head())

#### landscape viz of tf-force across pseudotime

In [ ]:
fig2 = waves.plot_landscape('BACH2', 'XBP1')
fig2.show()

# Softmax per force curve to assign wave

In [ ]:
# row_scaling example: {('NFKB1', 'AFF3'): 0.4} scales that link to 40% of its values
force_df_for_cluster, reg_labels, dtime, fig = waves.plot_force_heatmap(
    PB_links_plotting,
    cmap='RdBu_r',
    vmax=None,
    figsize=(3.5, 3),
    row_scaling=None,
)
plt.yticks(fontsize=10)
plt.tight_layout()
# plt.savefig(os.path.join(config.OUTPUT_FOLDER, 'enriched_links_PB_reordered.pdf'), bbox_inches='tight', dpi=300, format='pdf')
plt.show()

In [ ]:
# Softmax peak pseudotime per link (aggregated)
regulation_pseudotimes = waves.link_peak_pseudotimes(
    PB_links_plotting, top_k=5, temperature=1.0, method='weighted_mean'
)

In [ ]:
display(regulation_pseudotimes)

## Wave assignment by cell-state termination pseudotimes

In [ ]:
# PB -> 3 waves: switch boundaries are the termination pseudotimes of ActB-4 then earlyPB.
# A link lands in wave 1 if its softmax peak <= ActB-4 termination, wave 2 if between the
# two terminations, wave 3 if after earlyPB termination.
for state in ['ActB-4', 'earlyPB']:
    print(state, 'termination pseudotime:',
          ff.termination_pseudotime(state, PB_post_bifurcation_window_indices, method='threshold'))

pb_wave_assignments = waves.classify_waves_from_states(
    ff,
    PB_post_bifurcation_window_indices,
    boundary_states=['ActB-4', 'earlyPB'],
    termination_method='threshold',
    threshold_frac=0.1,
)
display(pb_wave_assignments)

# Wave Validation

## Overall trajectory FireFate links vs random links

In [ ]:
# Class 3: validation of wave-clustered links vs random links (skeleton — not yet implemented)
validation = WaveValidation(
    force_curves=waves.force_curves,
    dtime=waves.dtime,
    wave_assignments=pb_wave_assignments,
)
# validation.sample_random_links(n_per_wave=50)  # NotImplementedError — to be written
# validation.compare_by_wave()
# validation.plot()

## Wave-clustered FireFate links vs random links

In [ ]:
# GC fate -> 2 waves: single switch boundary at the termination pseudotime of ActB-3.
# (compute GC forces first, then classify against the GC post-bifurcation windows)
# gc_force_curves = waves.compute_forces(GC_links_plotting, varname='w_in')
# gc_wave_assignments = waves.classify_waves_from_states(
#     ff,
#     GC_post_bifurcation_window_indices,
#     boundary_states=['ActB-3'],
#     termination_method='threshold',
#     threshold_frac=0.1,
# )
# display(gc_wave_assignments)